# Lab 5 - Support Vector Machines 

The objective of this lab is to explore implementation and hyperparameter tuning for a Support Vector Machine (SVM). We will use a dataset of geochemical measurements from sediments collected off the west coast of Antarctica and look at how we might use SVM to correctly classify the region where the data was collected based on the chemistry. We will use this example to get a better understanding of how the choice of kernel, regularization parameter, and class weighting can affect the performance of the SVM classifier. 

First, let's import the packages we need to run our model

In [6]:
# load libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import svm
from sklearn import metrics
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.model_selection import GridSearchCV

The code block below imports the dataset will use and displays the first few rows of data. For our analysis today, we will focus on only two feature, `Ce` and `Sr/Th`, so that we can easily visualize the results. We will try to use these geochemical quantities to predict the `Sector` label - the region where the data was collected from. 

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/rtculberg/ml_in_eas/main/data/geochem_data.csv")
df.head()

The code below creates a scatterplot of longitude vs. latitude where each data point is colored by it's sector. You should see that we have data points from three distinct regions. Sector 1 corresponds to the Bellinghausen Sea, Sector 2 the Amundsen Sea, and Sector 3 to the Ross Sea off the coast of West Antarctica. 

In [ ]:
plt.scatter(df['Longitude'], df['Latitude'], c = df['Sector'])
plt.colorbar()
plt.show()

The code below creates scatterplot of `Ce` vs `Sr/Th` where each datapoint is colored by it's sector.     

Discuss with your group:       
(1) Do these two geochmical quantities seem to be useful for discriminating between sediments sourced from the different sectors?       
(2) Are there any aspects of this data that might be challenging for a classifier and why?    

In [ ]:
plt.scatter(df['Ce'], df['Sr/Th'], c = df['Sector'])
plt.show()

Create a new `x` variable that consists only of the `Ce` and `Sr/Th` measurements. Apply a Z-score transform to standardize the data using `StandardScaler()`. Call your new rescaled input features `x_scaled`. Create a new variable called `y` that contains the sector labels and convert it to a numpy array.

In [7]:
# Enter your code here

In the code block below, fill in the lines of code needed to instantiate a new `SVC` object and train the data on your entire data set. Of course, we would normally split the data into test and train sets, but in this lab, we are going to focus on gaining a better intuition for how different hyperparameters change the results and that will be easier to visualize on the whole dataset. Train your `SVC` model on the entire data set, then predict sector labels on that training data set. 

The code you have been provided plots the original data with sectors labels, as well as the predicted sector labels on side by side plots. Sector 1 is in light blue, sector 2 is in darker blue, and sector 3 is in pink. The third plot shows the decision boundaries (what class a data point would be assigned to based on it's location in the data space). The support vectors are circled in black.          

Once you have your code up and running, try the following scenarios to gain some intuition for how different hyperparameters will change the results.      

(1) Compare unbalanced and balanced class weights.     
(2) Compare `linear` and `rbf` kernels.         
(3) Using balanced class weights and an `rbf` kernel, try the following values for `C` (the regularization parameter): [1e-2, 1, 1e3]         
(4) Using balanced class weight, rbf kernel, and `C=1`, try the following values for `gamma`: [1e-2, 1, 1e3]      

Discuss with your group:     
(1) Is balancing the class weights important for this dataset or not? Why?        
(2) Which kernel seems to provide a better fit to the data? Why?     
(3) What is the impact of changing `C`, the regularization parameter? Is this consistent with your understanding of how regularization works?     
(4) What is the impact of changing `gamma`? How might this trade-off with changing `C`?

In [ ]:
# Enter code here to instantiate a new SVC model. Remember that you will need to specify the random_state, 
# whether to use balanced class weights, the type of kernel, the value for the regularization parameter C, 
# and (if you are using the rbf kernel) the value of the shape parameter gamma. 

# Enter code here to train and model and predict the class of each data point in the training data set


fig, (ax1, ax2, ax3) = plt.subplots(1,3,layout='constrained', figsize=(10,3))

ax1.scatter(x_scaled[:,0], x_scaled[:,1], c = y_pred, cmap='cool')
ax1.set_xlabel('Ce (scaled)')
ax1.set_ylabel('Sr/Th (scaled)')
ax1.set_title("Predicted Data Labels")

ax2.scatter(x_scaled[:,0], x_scaled[:,1], c = y, cmap='cool')
ax2.set_xlabel('Ce (scaled)')
ax2.set_ylabel('Sr/Th (scaled)')
ax2.set_title("True Data Labels")

DecisionBoundaryDisplay.from_estimator(model, x_scaled, response_method='predict', ax = ax3, cmap='cool')
ax3.scatter(x_scaled[:,0], x_scaled[:,1], c = y, edgecolor='k', cmap='cool')
ax3.scatter(model.support_vectors_[:,0], model.support_vectors_[:,1], s=100, facecolors='none', edgecolors='k')
ax3.set_title("Decision Boundaries")

plt.show()

A more reliable way to choose the best combination of hyperparameters is to implement a grid search, where we define a list of possible hyperparameter values and test all possible combinations to find the set that best fits our data. In Scikit-Learn, this implemented through the `GridSearchCV` function. The code block below shows you how to instantiate a new `GridSearchCV` object and pass it the `svc` model and the list of `parameters` that we want to test. Write a line of code to call the `fit()` function on your `GridSearchCV` object and pass it the input features and labels to run the grid search. 

In [ ]:
parameters = {'kernel':('linear', 'rbf'), 'C': np.logspace(-2,2,10), 'gamma': np.logspace(-2,2,10)}
svc = svm.SVC(class_weight='balanced')
clf = GridSearchCV(svc, parameters)
# Enter your code here to run GridSearchCV with the scaled input features (x) and labels (y)


Your `GridSearchCV` object has an attribute called `best_params_` that contains a list of the best parameters from the hyperparameter optimization process. Print out the best parameters.   

Discuss with your group:     
(1) How do these best parameters compare with what you found during your manual search? Any surprises?     

In [14]:
# Enter code here to print out a list of best parameters


{'C': 0.5994842503189409, 'gamma': 4.6415888336127775, 'kernel': 'rbf'}

In the code block below, instantiate a new `SVC` model using the best fit hyperparameter values from your grid search. Train your best model on all of the available data, then use your newly trained model to predict sector labels on the whole dataset. The provided code will plot the same three plots showing the differences between the true and predicted labels, as well as the decision boundaries and support vectors.    

Discuss with your group:      
(1) How well does the best classifier appear to perform?       
(2) How well do you think this classifier might generalize to other data points?

In [ ]:
# Enter code here to instantiate, train, and predict with your best model


fig, (ax1, ax2, ax3) = plt.subplots(1,3,layout='constrained', figsize=(10,3))

ax1.scatter(x_scaled[:,0], x_scaled[:,1], c = y_pred, cmap='cool')
ax1.set_xlabel('Ce (scaled)')
ax1.set_ylabel('Sr/Th (scaled)')
ax1.set_title("Predicted Data Labels")

ax2.scatter(x_scaled[:,0], x_scaled[:,1], c = y, cmap='cool')
ax2.set_xlabel('Ce (scaled)')
ax2.set_ylabel('Sr/Th (scaled)')
ax2.set_title("True Data Labels")

DecisionBoundaryDisplay.from_estimator(best_model, x_scaled, response_method='predict', ax = ax3, cmap='cool')
ax3.scatter(x_scaled[:,0], x_scaled[:,1], c = y, edgecolor='k', cmap='cool')
ax3.scatter(model.support_vectors_[:,0], model.support_vectors_[:,1], s=100, facecolors='none', edgecolors='k')
ax3.set_title("Decision Boundaries")

plt.show()